In [3]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, datasets, models

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

bbox = pd.read_csv("/content/drive/MyDrive/INTRO2AI/annotation.csv")
bbox.head(), bbox.shape

(         filename  xmin  ymin  xmax  ymax
 0  image_0000.jpg   534   360   663   504
 1  image_0001.jpg   400   249   595   573
 2  image_0002.jpg   474   254   678   468
 3  image_0003.jpg   422   347   546   475
 4  image_0004.jpg   222   458   447   675,
 (653, 5))

In [6]:
train_data_path = "/content/drive/MyDrive/INTRO2AI/train"
train_label_path = "/content/drive/MyDrive/INTRO2AI/train_label.csv"
train_labels = pd.read_csv(train_label_path)
train_dict = train_labels.to_dict(orient="records") # trả về một list các dict, với mỗi dict là 1 row trong train_labels

In [7]:
train_dict[0]

{'Unnamed: 0': 0, 'image_name': 'image_0000.jpg', 'label': 3}

In [8]:
class TrainImageDataset(Dataset):
    def __init__(self, data_dict, input_path, transform=None):
        self.data_dict = data_dict
        self.input_path = input_path
        self.transform = transform

    def __len__(self):
        return len(self.data_dict)

    def __getitem__(self, idx):
        img_name = self.data_dict[idx]['image_name'] # "image_xxxx.jpg"
        label = self.data_dict[idx]['label']

        img_path = os.path.join(self.input_path, img_name)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
class RandomCrop:
    """
    Crop randomly the image, but doesnt invade the bbox
    """
    def __call__(self, img)
    """
    img: an instance of class
    """

In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [10]:
BATCH_SIZE = 8
train_dataset = TrainImageDataset(data_dict=train_dict, input_path=train_data_path, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 15

In [12]:
# !!! Do not change anything of this cell !!!
from torch import Tensor
from typing import Type

class BasicBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        expansion: int = 1,
        downsample: nn.Module = None,
    ) -> None:
        super(BasicBlock, self).__init__()

        self.expansion = expansion
        self.downsample = downsample
        self.conv1_layer = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )

        self.batch_norm1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2_layer = nn.Conv2d(
            out_channels,
            out_channels * self.expansion,
            kernel_size=3,
            padding=1,
            bias=False,
        )

        self.batch_norm2 = nn.BatchNorm2d(out_channels * self.expansion)

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1_layer(x)
        out = self.batch_norm1(out)
        out = self.relu(out)

        out = self.conv2_layer(out)
        out = self.batch_norm2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)
        return out


class CNN(nn.Module):
    def __init__(
        self,
        block: Type[BasicBlock],
        img_channels: int = 3,
        num_classes: int = 10,
    ) -> None:
        super(CNN, self).__init__()
        layers = [2, 2, 2, 2]
        self.expansion = 1

        self.in_channels = 64

        self.conv_layer = nn.Conv2d(
            in_channels=img_channels,
            out_channels=self.in_channels,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )
        self.batch_norm = nn.BatchNorm2d(self.in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool_layer = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer_1 = self._make_layer(block, 64, layers[0])
        self.layer_2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer_3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer_4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool_layer = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_layer = nn.Linear(512 * self.expansion, num_classes)

    def _make_layer(
        self, block: Type[BasicBlock], out_channels: int, blocks: int, stride: int = 1
    ) -> nn.Sequential:
        downsample = None
        if stride != 1:
            downsample = nn.Sequential(
                nn.Conv2d(
                    self.in_channels,
                    out_channels * self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels * self.expansion),
            )
        layers = []
        layers.append(
            block(self.in_channels, out_channels, stride, self.expansion, downsample)
        )
        self.in_channels = out_channels * self.expansion

        for i in range(1, blocks):
            layers.append(
                block(self.in_channels, out_channels, expansion=self.expansion)
            )
        return nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        x = self.conv_layer(x)
        x = self.batch_norm(x)
        x = self.relu(x)
        x = self.maxpool_layer(x)

        x = self.layer_1(x)
        x = self.layer_2(x)
        x = self.layer_3(x)
        x = self.layer_4(x)

        x = self.avgpool_layer(x)
        x = torch.flatten(x, 1)
        x = self.fc_layer(x)

        return x

model = CNN(block=BasicBlock, num_classes=NUM_CLASSES).to(device)

In [13]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [14]:
from timeit import default_timer as timer

model_results = {"train_loss": [], "train_acc": []}
EPOCHS = 20
start_timer = timer()

for epoch in tqdm(range(EPOCHS)):
    model.train()
    train_loss, train_acc = 0, 0
    for batch, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(X)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)
        train_loss += loss.item()

    train_loss /= len(train_dataloader)
    train_acc /= len(train_dataloader)
    model_results["train_loss"].append(train_loss)
    model_results["train_acc"].append(train_acc)

    torch.save(model.state_dict(), f"model_epoch_{epoch}.pth")

    print(f"Epoch: {epoch+1} | Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}")

end_timer = timer()
print(f"Total training time: {end_timer - start_timer:.3f} seconds")

  0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
test_data_path = ".../test"

In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, input_path, transform=None):
        self.input_path = input_path
        self.img_names = os.listdir(input_path)
        self.transform = transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.input_path, self.img_names[idx])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, img_path

In [ ]:
BATCH_SIZE = 1
test_dataset = TestImageDataset(input_path=test_data_path, transform=transform)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
image_names = []
all_preds = []
all_labels = []

model.load_state_dict(torch.load(f"model_epoch_{EPOCHS - 1}.pth"))
model.eval()

with torch.no_grad():
    for images, paths in test_dataloader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        image_names.extend([os.path.basename(path) for path in paths])
        all_preds.extend(predicted.cpu().numpy())

df = pd.DataFrame({"image_name": image_names, "label": all_preds})

df.to_csv("prediction.csv", index=False)